# MCP and LangChain Tool Integration (Chapter 8)

This notebook accompanies Chapter 8 §8.2.5 of *Context Engineering with DSPy*. You will connect to a Model Context Protocol (MCP) server with `mcp.ClientSession`, convert its tools into DSPy tools via `dspy.Tool.from_mcp_tool`, and hand them to a `dspy.ReAct` agent. The last section shows the equivalent one-liner for adopting existing LangChain tools via `dspy.Tool.from_langchain`.

**Required environment variable**

- `OPENAI_API_KEY` — used by the default `openai/gpt-5-mini` LM.

**Service dependency: an MCP server**

You need a running MCP server. We do **not** ship one in this repo — read the spec and the SDK quickstarts at <https://modelcontextprotocol.io> and pick the server most relevant to your task (filesystem, GitHub, search, your own custom server, etc.). The example below assumes your server is launched as `python mcp_server.py` over stdio; adjust `StdioServerParameters` to match whatever command starts your server. If no server is reachable, the cell will print a clear error instead of crashing the notebook.

In [ ]:
%pip install -r ../requirements.txt -q

In [ ]:
%pip install mcp -q

In [ ]:
import os
import dspy
from dotenv import load_dotenv

load_dotenv()

dspy.configure(lm=dspy.LM("openai/gpt-5-mini"))

## Connecting to an MCP server

MCP is asynchronous, so we use `asyncio` plus `mcp.ClientSession` to list the server's tools and convert each one to a DSPy tool. The agent itself must be awaited with `agent.acall(...)` because tool execution is async under the hood.

In [ ]:
import asyncio

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


async def build_agent():
    server_params = StdioServerParameters(
        command="python",
        args=["mcp_server.py"],
    )

    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # Convert all MCP tools to DSPy tools
            mcp_tools = await session.list_tools()
            tools = [
                dspy.Tool.from_mcp_tool(session, tool)
                for tool in mcp_tools.tools
            ]

            # Build and run the agent
            agent = dspy.ReAct(
                "user_request -> response",
                tools=tools,
                max_iters=10,
            )

            result = await agent.acall(
                user_request="Book a flight from SFO to JFK on September 1st"
            )
            return result


try:
    result = asyncio.run(build_agent())
    print(result)
except Exception as exc:
    print(
        "Could not reach an MCP server. Start your MCP server (see "
        "https://modelcontextprotocol.io) and adjust StdioServerParameters above, then re-run.\n"
        f"Underlying error: {exc!r}"
    )

## Bringing existing LangChain tools into DSPy

If your team already maintains LangChain tools you do not have to rewrite them. Wrap any `@langchain.tools.tool` callable with `dspy.Tool.from_langchain` and pass the result alongside your native DSPy tools.

In [ ]:
%pip install langchain langchain-core -q

In [ ]:
from langchain.tools import tool as lc_tool


@lc_tool
def web_search(query: str) -> str:
    """Search the web for information."""
    return f"[stub] web hits for: {query}"


dspy_tool = dspy.Tool.from_langchain(web_search)
print(dspy_tool)